## Exercise on STA computation and Linear-NonLinear model fitting

<br>A small exercise on STA and LNP model fitting, by U. Ferrari with the help of M. Chalk.</br>
<b>Goal:</b> to compute the STA of a single Retinal Ganglion cell from the response to a set of checkerboad images. 

Complete the missing code in the marked cells. Note that the code is not intended to be optimized for computational performance. 
In particular, most of the for cycles can be rewrittenm as vector multiplications 
which in Matlab are usually much faster. <br>
<i>Brave students can give it a try, and optimize the code</i>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import os


## The data
Data consist in a set of checkerboard images and the response of a ganglion cell.
Before even loading the data, it's good practice to start by looking at the file content.

In [ ]:
# What's in there? Which dimensions have the vectors?

mat_contents = sio.whosmat('TD_Ferrari.mat')

for name, shape, dtype in mat_contents:
    print(f"Name: {name}, Shape: {shape}, Type: {dtype}")

### Introduction: always start by having a look at the raw data

In [ ]:
# Fill in the blanks

# To load the data file you need to be in the right folder (where the
# data are located
data = sio.loadmat('TD_Ferrari.mat')

# Extract stimulus and response
stimulus = ...
response = ...

# Get the size of the stimulus
# nx = size of image patch in one direction
# N =  number of images
nx, _, N = ...

In [ ]:
# Fill in the blanks

# Plot the first checkerboard image
plt.figure(figsize=(8, 6))
plt.imshow(..., cmap='gray')
plt.title('Example stimulus')
plt.colorbar()
plt.tight_layout()
plt.show()

# Plot the histogram of the response
plt.figure(figsize=(8, 6))
plt.hist(..., bins=100, edgecolor='black')
plt.xlabel('Recorded spike counts')
plt.ylabel('Number')
plt.title('Histogram of spike counts')
plt.tight_layout()
plt.show()

## Compute the Spike Triggered Average (STA)

In [ ]:
# Fill in the blanks

# Define the sta matrix
sta = np.zeros((nx, nx))

# To be completed

# Define the sta matrix
sta = np.zeros((nx, nx))

# Compute the average of the images weighting on the response
# [[Can you optimize this?]]
for n in range(N):
    sta = sta + ...

# Don't forget to normalize...

### Plot the STA

In [ ]:
# Fill in the blanks

# Plot the sta
plt.figure(figsize=(8, 6))
plt.imshow(..., cmap='coolwarm', vmin = -1, vmax = 1)
plt.title('STA (Spike Triggered Average)')
plt.colorbar()
plt.tight_layout()
plt.show()

## Compute the filtered stimulus 'signal'
<b>Extra points: </b><i>Can you optimize this?</i>

In [ ]:
# Fill in the blanks

signal = np.zeros(response.shape)
for n in range(N):
    signal[n] = np.sum( ... )

### Plot signal versus response

In [ ]:
# Fill in the blanks

plt.figure(figsize=(8, 6))
plt.plot(..., ..., 'k.', markersize=2)
plt.ylim([0, np.max(response) * 1.05])
plt.xlabel('signal = sta * x')
plt.ylabel('spike count')
plt.title('Filter versus spikes')
plt.tight_layout()
plt.show()

### Compute the anchor points of the non-linearity

Bin the signal in 50 bins.
Hint: 


    COUNT, EDGES = np.histogram(signal, bins=nBin)
    BIN = np.digitize(signal, EDGES)

where:
- COUNTS = vector of size nBin counting the number of points per bin
- EDGES  = vector of size nBin+1 with the edges of the bins
- BIN = vector of the same size as signal, with the bin number where each element fall: BIN(n) = k if signal(n) falls in the kth bin


In [ ]:
# Fill in the blanks

nBin = ...
COUNT, EDGES = np.histogram(signal, bins=nBin)
BIN          = np.digitize(signal, EDGES)

# Compute the middle points between the EDGES
anchor_x = EDGES[:-1] + (EDGES[1:] - EDGES[:-1]) / 2

# Compute the average of the response within each bin
anchor_y = np.zeros(nBin)
for bin_idx in range(1, nBin + 1):
    if np.any(BIN == bin_idx):
        anchor_y[bin_idx - 1] = ...

# CubicSpline allows for constructing a cubic interpolation between (x,y) points
from   scipy.interpolate import CubicSpline
nonlinearity  = CubicSpline( ..., ...)

# Evaluate the spline interpolation at signal points
predictions   = nonlinearity( ... )

#### Plot signal versus response with anchor points

In [ ]:
# Fill in the blanks

plt.figure(figsize=(8, 6))
plt.plot(signal, response, 'k.', markersize=2, alpha=0.5)
plt.plot(...,..., 'r.', markersize=18)
plt.scatter(..., ..., c = 'blue')
plt.ylim([0, np.max(response) * 1.05])
plt.xlabel('signal = sta * x')
plt.ylabel('spike count')
plt.title('Filter versus spikes with anchor points')
plt.tight_layout()
plt.show()


# Scatterplot between the responses for all images and the non-linear predictions
plt.figure(figsize=(8, 6))
plt.plot(..., ..., 'k.', markersize=14, alpha=0.5)
plt.plot([0, 120], [0, 120], 'k--', linewidth=2)
plt.ylim([0, np.max(response) * 1.05])
plt.xlim([0, np.max(predictions) * 1.05])
plt.ylabel('Predicted mean spike count')
plt.xlabel('Observed spike count')
plt.title('Spike count versus prediction')
plt.tight_layout()
plt.show()

## Fit a linear-nonlinear-Poisson (LNP) model

In [ ]:
# Fill in the blanks

# Initialize parameters
b = 0  # initial bias term
w = 1e-9 * np.random.randn(nx**2)  # initial linear weights

# Reshape stimulus for matrix operations
# Xtilde will be (nx^2, N) - each column is a flattened image
Xtilde = stimulus.reshape(nx**2, N)

# Parameters for gradient descent
Nit = 300  # number of iterations
eta = 1e-2  # step size

L = np.zeros(Nit)  # log likelihood

for i in range(Nit):
    # Firing rate (predicted)
    # f = ?  # ex. 2(a) predicted firing rate on each trial (vector of size N)
    # Hint: f = exp(w^T * X + b)
    f = ...
    
    # Log likelihood (Poisson)
    L[i] = (np.log(f) @ response - np.sum(f)) / N
    
    # Derivative of log likelihood
    dL_w = Xtilde @ (response - f) / N
    dL_b = np.sum(response - f) / N
    
    # Update parameters
    # gradient descent updates of w and b
    # w = ?  
    # b = ?

#### Plot loss function versus iterations

In [ ]:
# Fill in the blanks

plt.figure(figsize=(8, 6))
plt.plot(..., ...)
plt.xlabel('Iterations')
plt.ylabel('Log likelihood')
plt.title('Loss function')
plt.tight_layout()
plt.show()


#### Plot learned filter

In [ ]:
# Fill in the blanks

# 2D plot of the inferred filter
plt.figure(figsize=(8, 6))
plt.imshow(... , cmap='coolwarm',vmin = -0.1, vmax = 0.1)
plt.title('Learned filter')
plt.colorbar()
plt.tight_layout()
plt.show()

#### Plot spike count versus model prediction

In [ ]:
# Fill in the blanks

# Estimate model prediction
f_final = ...

# Compare predicted and observed spike count
plt.figure(figsize=(8, 6))
plt.plot(..., ..., 'k.', alpha=0.5)
plt.plot([0, 120], [0, 120], 'k--', linewidth=2)
plt.xlabel('Predicted mean spike count')
plt.ylabel('Observed spike count')
plt.title('LNP model: Spike count versus prediction')
plt.tight_layout()
plt.show()

<b>Which apprach does work better? STA or LNP fit?</b>

This is left as exercize. Remember that for fair comparison, you would need to split your data into training and testing.
You can then estimate performance with correlation between data and prediction.


## Exercise completed! :)